## 1.熵权法计算主题得分



In [188]:
import pandas as pd
import csv
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import MinMaxScaler

### （1）先从文档拿到所有推文数据，整理为表格吧

In [189]:
#把每列数据整理起来创建：
fans = []
vote = []
comment = []
for i in range(1,51):
    with open(f"../reddit/content/reddit_content_{i}.txt",'r',encoding="utf-8") as f:
        line = f.readline().split("\t",maxsplit=3)[:3]
        print(line)
        fans.append(int(line[0].split(' ')[2]))
        vote.append(int(line[1].split(' ')[2]))
        comment.append(int(line[2].split(' ')[2]))
print(fans)
print(vote)
print(comment)

['浏览量 : 3605980', '点赞量 : 11', '评论量 : 30']
['浏览量 : 3605981', '点赞量 : 3553', '评论量 : 92']
['浏览量 : 3605982', '点赞量 : 1381', '评论量 : 268']
['浏览量 : 3605985', '点赞量 : 395', '评论量 : 55']
['浏览量 : 3605983', '点赞量 : 3432', '评论量 : 245']
['浏览量 : 3605985', '点赞量 : 885', '评论量 : 83']
['浏览量 : 3605987', '点赞量 : 610', '评论量 : 45']
['浏览量 : 3605985', '点赞量 : 6811', '评论量 : 361']
['浏览量 : 3605983', '点赞量 : 0', '评论量 : 90']
['浏览量 : 612367', '点赞量 : 11060', '评论量 : 295']
['浏览量 : 612367', '点赞量 : 4552', '评论量 : 265']
['浏览量 : 612367', '点赞量 : 3456', '评论量 : 249']
['浏览量 : 612367', '点赞量 : 3008', '评论量 : 190']
['浏览量 : 612365', '点赞量 : 2726', '评论量 : 303']
['浏览量 : 612364', '点赞量 : 2672', '评论量 : 236']
['浏览量 : 612364', '点赞量 : 2470', '评论量 : 571']
['浏览量 : 612357', '点赞量 : 2237', '评论量 : 112']
['浏览量 : 612342', '点赞量 : 1835', '评论量 : 198']
['浏览量 : 612344', '点赞量 : 1779', '评论量 : 197']
['浏览量 : 612345', '点赞量 : 1597', '评论量 : 153']
['浏览量 : 138869', '点赞量 : 887', '评论量 : 203']
['浏览量 : 612254', '点赞量 : 616', '评论量 : 108']
['浏览量 : 612254', '点赞量 : 457', '评论量 : 2

In [190]:
idx = [f'推文{i}' for i in range(1,51)]
Indicator_df = pd.DataFrame({
    'fans':fans,
    'vote':vote,
    'comment':comment
},index=idx)
print(Indicator_df)


          fans   vote  comment
推文1    3605980     11       30
推文2    3605981   3553       92
推文3    3605982   1381      268
推文4    3605985    395       55
推文5    3605983   3432      245
推文6    3605985    885       83
推文7    3605987    610       45
推文8    3605985   6811      361
推文9    3605983      0       90
推文10    612367  11060      295
推文11    612367   4552      265
推文12    612367   3456      249
推文13    612367   3008      190
推文14    612365   2726      303
推文15    612364   2672      236
推文16    612364   2470      571
推文17    612357   2237      112
推文18    612342   1835      198
推文19    612344   1779      197
推文20    612345   1597      153
推文21    138869    887      203
推文22    612254    616      108
推文23    612254    457      289
推文24   8083465    725      183
推文25    612255    595      134
推文26    446941    259      162
推文27   2260965     43      106
推文28    612254    371      117
推文29    612254   1732      556
推文30    612254    436      111
推文31    612254    725      258
推文32    

In [191]:
#或者使用列表＋元组创建：
data_list = []
for i in range(1,1274):
    with open(f"../reddit/massive_content/reddit_content_{i}.txt",'r',encoding="utf-8") as f:
        line = f.readline().split("\t",maxsplit=3)[:3]
        # print(line)
        # print(i)
        data = (int(line[0].split(" ")[2]),int(line[1].split(" ")[2]),int(line[2].split(" ")[2]))
        print(data)
        data_list.append(data)
# print(data_list)
origin_df = pd.DataFrame(data_list,index=[f'推文{i}' for i in range(1,1274)],columns=['fans','vote','comment'])
# print(df.head())
# print(df.info())
#修改单个值 一个先行后列一个先列后行
# df.loc['推文9','vote'] = 1
#df['vote'].loc['推文9'] = 1
origin_df


(611779, 66, 63)
(611779, 841, 94)
(611779, 5, 0)
(611779, 39, 14)
(611779, 105, 28)
(611779, 46, 12)
(611779, 8530, 1001)
(611779, 7506, 619)
(611779, 7447, 126)
(611779, 6655, 150)
(611779, 6549, 182)
(611779, 6522, 312)
(611779, 6485, 144)
(611779, 6446, 184)
(611779, 6373, 193)
(611779, 5761, 702)
(611779, 5726, 1276)
(611779, 5560, 529)
(611779, 5546, 318)
(611779, 5233, 131)
(611779, 5031, 177)
(611779, 4976, 1047)
(611779, 4918, 214)
(611779, 4809, 335)
(611779, 4724, 48)
(611779, 4609, 15)
(611779, 4556, 265)
(611779, 4454, 305)
(611779, 4337, 165)
(611779, 4226, 205)
(611779, 4168, 222)
(611779, 4074, 281)
(611779, 3950, 788)
(611779, 3949, 185)
(611779, 3754, 399)
(611779, 3740, 240)
(611779, 3616, 65)
(611779, 3587, 240)
(611779, 3586, 314)
(611779, 3542, 117)
(611779, 3510, 81)
(611779, 3487, 559)
(611779, 3468, 181)
(611779, 3459, 175)
(611779, 3457, 249)
(611779, 3417, 240)
(611779, 3380, 216)
(611779, 3369, 699)
(611779, 3291, 409)
(611779, 3265, 266)
(611779, 3245, 206)

,fans,vote,comment
推文1,611779,66,63
推文2,611779,841,94
推文3,611779,5,0
推文4,611779,39,14
推文5,611779,105,28
...,...,...,...
推文1269,101521,7,19
推文1270,101521,7,10
推文1271,787,1,0
推文1272,101521,39,10


In [192]:
hot_sr = pd.Series(origin_df['vote'] + origin_df["comment"],name='综合热度')
hot_sr

推文1       129
推文2       935
推文3         5
推文4        53
推文5       133
         ... 
推文1269     26
推文1270     17
推文1271      1
推文1272     49
推文1273      1
Name: 综合热度, Length: 1273, dtype: int64

### (2)整理原始数据，处理缺失值、异常值、统一格式

In [193]:
origin_df.loc[origin_df['fans']==0,'fans'] = int(origin_df['fans'].mean())
origin_df.loc[origin_df['vote']==0,'vote'] = int(origin_df['vote'].mean())
origin_df.loc[origin_df['comment']==0,'comment'] = int(origin_df['comment'].mean())
# origin_df.loc[origin_df['fans']==0,'fans']
origin_df.loc["推文1274","fans"] = origin_df["fans"].mean()
origin_df.loc["推文1274","vote"] = origin_df["vote"].mean()
origin_df.loc["推文1274","comment"] = origin_df["comment"].mean()
fan_sr = origin_df['fans']
# origin_df.drop(columns='fans',inplace=True)
origin_df

,fans,vote,comment
推文1,6.117790e+05,66.000000,63.000000
推文2,6.117790e+05,841.000000,94.000000
推文3,6.117790e+05,5.000000,107.000000
推文4,6.117790e+05,39.000000,14.000000
推文5,6.117790e+05,105.000000,28.000000
...,...,...,...
推文1270,1.015210e+05,7.000000,10.000000
推文1271,7.870000e+02,1.000000,107.000000
推文1272,1.015210e+05,39.000000,10.000000
推文1273,7.870000e+02,1.000000,107.000000


### （3）数据归一化

In [194]:
#直接使用sklearn库
scaler = MinMaxScaler()
origin_df_normalized = pd.DataFrame(scaler.fit_transform(origin_df),columns=origin_df.columns,index=origin_df.index)
print(origin_df_normalized)

            fans      vote   comment
推文1     0.015121  0.001097  0.006266
推文2     0.015121  0.014174  0.009400
推文3     0.015121  0.000067  0.010714
推文4     0.015121  0.000641  0.001314
推文5     0.015121  0.001755  0.002729
...          ...       ...       ...
推文1270  0.002508  0.000101  0.000910
推文1271  0.000019  0.000000  0.010714
推文1272  0.002508  0.000641  0.000910
推文1273  0.000019  0.000000  0.010714
推文1274  0.032167  0.021529  0.011071

[1274 rows x 3 columns]


In [195]:
#或者进行手动归一化
# for column in origin_df.columns:
#     max_val = origin_df[column].max()
#     min_val = origin_df[column].min()
#     origin_df[column] = (origin_df[column] - min_val) / (max_val - min_val) # 注意这行，origin_df[column] - min_val这是一个向量化操作，pandas 会自动对 Series 中的每个元素应用减法。
#     print(type(origin_df[column]))
# print(origin_df.head())


### （4）计算信息熵

In [196]:
def calculate_entropy(column, bins=10):
    """
    计算一列数据的信息熵（通过分箱处理连续数据）

    参数:
    - column: pandas Series，输入的列数据
    - bins: int，分箱数量，默认10

    返回:
    - entropy: float，信息熵值
    """
    # 分箱统计频率
    counts, _ = np.histogram(column, bins=bins)
    probabilities = counts / counts.sum()  # 计算概率

    # 避免概率为0导致log2(0)错误，添加极小值1e-10
    entropy = -np.sum(probabilities * np.log2(probabilities + 1e-10))
    return entropy


# 计算各列的信息熵（默认分10箱）
entropies = origin_df_normalized.apply(calculate_entropy, bins=10)

# 输出结果
print("各特征的信息熵：")
print(entropies)

各特征的信息熵：
fans       0.166718
vote       0.222944
comment    0.083040
dtype: float64


### （5）计算归一化权重

In [197]:
def entropy_weight(df, bins=10):
    """
    基于信息熵计算特征的归一化权重

    参数:
    - df: pandas DataFrame，输入数据（需已标准化）
    - bins: int，分箱数量，默认10

    返回:
    - weights: pandas Series，归一化后的权重
    """
    # 计算差异系数（差异系数 = 1 - 熵）
    differences = 1 - entropies

    # 处理边界情况：所有差异系数为0时（熵全为1），直接返回均匀权重
    if (differences == 0).all():
        return pd.Series([1/len(df.columns)] * len(df.columns), index=df.columns)

    # 归一化权重
    weights = differences / differences.sum()
    return weights

# 计算归一化权重
weights = entropy_weight(entropies, bins=10)

# 输出结果
print("各特征的归一化权重：")
print(weights.round(4))  # 保留4位小数

各特征的归一化权重：
fans       0.3297
vote       0.3075
comment    0.3628
dtype: float64


In [198]:
tail_weights = weights.copy().iloc[[1,2]]

print(tail_weights)
origin_df_normalized['综合热度'] = hot_sr
# df = pd.concat([fan_sr,origin_df],axis=1)
origin_df_normalized

vote       0.307465
comment    0.362822
dtype: float64


,fans,vote,comment,综合热度
推文1,0.015121,0.001097,0.006266,129.0
推文2,0.015121,0.014174,0.009400,935.0
推文3,0.015121,0.000067,0.010714,5.0
推文4,0.015121,0.000641,0.001314,53.0
推文5,0.015121,0.001755,0.002729,133.0
...,...,...,...,...
推文1270,0.002508,0.000101,0.000910,17.0
推文1271,0.000019,0.000000,0.010714,1.0
推文1272,0.002508,0.000641,0.000910,49.0
推文1273,0.000019,0.000000,0.010714,1.0


## 2.LDA得出每个推文主题概率

### （1）先是从LDA那边拿到主题概率的二维列表，把它加入到表格里，再合并相同主题

In [199]:
with open('topic_probability', 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        data = [row for row in reader]
data

[['(0, 0.00012110762)',
  '(1, 0.22770548)',
  '(2, 0.084483884)',
  '(3, 0.00011003951)',
  '(4, 6.336565e-05)',
  '(5, 6.195557e-05)',
  '(6, 5.2807743e-05)',
  '(7, 0.019544618)',
  '(8, 6.405703e-05)',
  '(9, 0.66766965)',
  '(10, 5.131483e-05)',
  '(11, 7.173246e-05)'],
 ['(0, 0.04493776)',
  '(1, 0.025099125)',
  '(2, 3.0702016e-05)',
  '(3, 0.027011039)',
  '(4, 0.10338542)',
  '(5, 0.2684541)',
  '(6, 0.0050750063)',
  '(7, 1.4711723e-05)',
  '(8, 2.397468e-05)',
  '(9, 0.51754546)',
  '(10, 1.9205645e-05)',
  '(11, 0.008403493)'],
 ['(0, 0.13404542)',
  '(1, 0.14467877)',
  '(2, 0.09079669)',
  '(3, 0.121796414)',
  '(4, 0.07013633)',
  '(5, 0.068575576)',
  '(6, 0.0584503)',
  '(7, 0.043507755)',
  '(8, 0.07090159)',
  '(9, 0.060916163)',
  '(10, 0.056797866)',
  '(11, 0.07939714)'],
 ['(0, 0.07742487)',
  '(1, 0.00064231595)',
  '(2, 0.00040307496)',
  '(3, 0.8741019)',
  '(4, 0.045481082)',
  '(5, 0.00030442848)',
  '(6, 0.0002594792)',
  '(7, 0.00019314454)',
  '(8, 0.0003

In [200]:
def parse_tuple_str(tuple_str):
    """将单个字符串'(a, b)'转换为数值元组(int, float)"""
    # 去除括号和空格，分割两个值
    clean_str = tuple_str.strip('()').replace(' ', '')
    id_str, value_str = clean_str.split(',')
    return (int(id_str), float(value_str))

# 转换整个数据结构
converted_data = [
    [parse_tuple_str(item) for item in sublist]
    for sublist in data
]

# # 验证输出
# print("转换结果：")
# for i, sublist in enumerate(converted_data):
#     print(f"文档{i+1}:")
#     for topic_id, value in sublist:
#         print(f"  主题{topic_id}: {value:.8f}")  # 保留8位小数
#     print()

# 如果需要保持原始结构：
print("完整数据结构：")
print(converted_data)

完整数据结构：
[[(0, 0.00012110762), (1, 0.22770548), (2, 0.084483884), (3, 0.00011003951), (4, 6.336565e-05), (5, 6.195557e-05), (6, 5.2807743e-05), (7, 0.019544618), (8, 6.405703e-05), (9, 0.66766965), (10, 5.131483e-05), (11, 7.173246e-05)], [(0, 0.04493776), (1, 0.025099125), (2, 3.0702016e-05), (3, 0.027011039), (4, 0.10338542), (5, 0.2684541), (6, 0.0050750063), (7, 1.4711723e-05), (8, 2.397468e-05), (9, 0.51754546), (10, 1.9205645e-05), (11, 0.008403493)], [(0, 0.13404542), (1, 0.14467877), (2, 0.09079669), (3, 0.121796414), (4, 0.07013633), (5, 0.068575576), (6, 0.0584503), (7, 0.043507755), (8, 0.07090159), (9, 0.060916163), (10, 0.056797866), (11, 0.07939714)], [(0, 0.07742487), (1, 0.00064231595), (2, 0.00040307496), (3, 0.8741019), (4, 0.045481082), (5, 0.00030442848), (6, 0.0002594792), (7, 0.00019314454), (8, 0.00031475435), (9, 0.00027042592), (10, 0.0002521435), (11, 0.00035246878)], [(0, 0.00018018512), (1, 0.00019448393), (2, 0.14360432), (3, 0.00016371706), (4, 9.4275616e-0

 ### （2）把这12个主题合并到表上

In [201]:
f_list = []
for line in converted_data:
    each_row = (line[0][1],line[1][1],line[2][1],line[3][1],line[4][1],line[5][1],line[6][1],line[7][1],line[8][1],line[9][1],line[10][1],line[11][1])
    f_list.append(each_row)
print(f_list)

[(0.00012110762, 0.22770548, 0.084483884, 0.00011003951, 6.336565e-05, 6.195557e-05, 5.2807743e-05, 0.019544618, 6.405703e-05, 0.66766965, 5.131483e-05, 7.173246e-05), (0.04493776, 0.025099125, 3.0702016e-05, 0.027011039, 0.10338542, 0.2684541, 0.0050750063, 1.4711723e-05, 2.397468e-05, 0.51754546, 1.9205645e-05, 0.008403493), (0.13404542, 0.14467877, 0.09079669, 0.121796414, 0.07013633, 0.068575576, 0.0584503, 0.043507755, 0.07090159, 0.060916163, 0.056797866, 0.07939714), (0.07742487, 0.00064231595, 0.00040307496, 0.8741019, 0.045481082, 0.00030442848, 0.0002594792, 0.00019314454, 0.00031475435, 0.00027042592, 0.0002521435, 0.00035246878), (0.00018018512, 0.00019448393, 0.14360432, 0.00016371706, 9.4275616e-05, 0.62368816, 7.856753e-05, 5.8482106e-05, 0.036492202, 0.19526254, 7.634636e-05, 0.000106723775), (0.00037182137, 0.0004013367, 0.028733829, 0.8976414, 0.00019454467, 0.00019021545, 0.00016212987, 0.000120682125, 0.00019666733, 0.07160958, 0.00015754634, 0.00022023237), (1.7173

In [202]:
index = [f'推文{j}' for j in range(1,1275)]
print(index)
pb = pd.DataFrame(f_list,columns=[f"主题{i}" for i in range(1,13)],index=index)
pb

['推文1', '推文2', '推文3', '推文4', '推文5', '推文6', '推文7', '推文8', '推文9', '推文10', '推文11', '推文12', '推文13', '推文14', '推文15', '推文16', '推文17', '推文18', '推文19', '推文20', '推文21', '推文22', '推文23', '推文24', '推文25', '推文26', '推文27', '推文28', '推文29', '推文30', '推文31', '推文32', '推文33', '推文34', '推文35', '推文36', '推文37', '推文38', '推文39', '推文40', '推文41', '推文42', '推文43', '推文44', '推文45', '推文46', '推文47', '推文48', '推文49', '推文50', '推文51', '推文52', '推文53', '推文54', '推文55', '推文56', '推文57', '推文58', '推文59', '推文60', '推文61', '推文62', '推文63', '推文64', '推文65', '推文66', '推文67', '推文68', '推文69', '推文70', '推文71', '推文72', '推文73', '推文74', '推文75', '推文76', '推文77', '推文78', '推文79', '推文80', '推文81', '推文82', '推文83', '推文84', '推文85', '推文86', '推文87', '推文88', '推文89', '推文90', '推文91', '推文92', '推文93', '推文94', '推文95', '推文96', '推文97', '推文98', '推文99', '推文100', '推文101', '推文102', '推文103', '推文104', '推文105', '推文106', '推文107', '推文108', '推文109', '推文110', '推文111', '推文112', '推文113', '推文114', '推文115', '推文116', '推文117', '推文118', '推文119', '推文120', '推文121', '推文122', '推文123', 

,主题1,主题2,主题3,主题4,主题5,主题6,主题7,主题8,主题9,主题10,主题11,主题12
推文1,0.000121,0.227705,0.084484,0.000110,0.000063,0.000062,0.000053,0.019545,0.000064,0.667670,0.000051,0.000072
推文2,0.044938,0.025099,0.000031,0.027011,0.103385,0.268454,0.005075,0.000015,0.000024,0.517545,0.000019,0.008403
推文3,0.134045,0.144679,0.090797,0.121796,0.070136,0.068576,0.058450,0.043508,0.070902,0.060916,0.056798,0.079397
推文4,0.077425,0.000642,0.000403,0.874102,0.045481,0.000304,0.000259,0.000193,0.000315,0.000270,0.000252,0.000352
推文5,0.000180,0.000194,0.143604,0.000164,0.000094,0.623688,0.000079,0.000058,0.036492,0.195263,0.000076,0.000107
...,...,...,...,...,...,...,...,...,...,...,...,...
推文1270,0.000220,0.587082,0.000149,0.065260,0.000115,0.000112,0.000096,0.146886,0.011532,0.177466,0.010952,0.000130
推文1271,0.134045,0.144679,0.090797,0.121796,0.070136,0.068576,0.058450,0.043508,0.070902,0.060916,0.056798,0.079397
推文1272,0.000392,0.826439,0.000266,0.132477,0.000205,0.000201,0.000171,0.000127,0.039145,0.000178,0.000166,0.000232
推文1273,0.134045,0.144679,0.090797,0.121796,0.070136,0.068576,0.058450,0.043508,0.070902,0.060916,0.056798,0.079397


In [203]:
#合并划分后的主题
# 定义合并规则（列索引和对应的新列名）
merge_rules = {
    "核心文化": ["主题5","主题7","主题12","主题1"],
    "载体表现": ["主题6","主题8","主题3","主题2"],
    "支撑机制":["主题9","主题11"],
    "泛化讨论":["主题4","主题10"]
}

# 循环处理每个合并规则
for new_col, cols_to_merge in merge_rules.items():
    # 生成新列（按行求和）
    pb[new_col] = pb[cols_to_merge].sum(axis=1)
    # 删除原列
    pb.drop(columns=cols_to_merge, inplace=True)

# 查看结果
print("\n合并后的数据：")
print(pb.head())


合并后的数据：
         核心文化      载体表现      支撑机制      泛化讨论
推文1  0.000309  0.331796  0.000115  0.667780
推文2  0.161802  0.293599  0.000043  0.544556
推文3  0.342029  0.347559  0.127699  0.182713
推文4  0.123518  0.001543  0.000567  0.874372
推文5  0.000460  0.767545  0.036569  0.195426


## 4.拿高频词表

In [204]:
num_word_df = pd.read_csv('./word_df.csv').drop(columns='Unnamed: 0',axis=1)
num_word_df

,spoiler,post,game,book,nezha,movie,soul,tang,character,rule,...,castorice,ban,might,someone,translation,wait,removed,amazing,definitely,shit
0,0.0,0.002232,0.033482,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.001116,0.0,0.002232,0.005580,0.000000,0.000000,0.000000,0.000000
1,0.0,0.000862,0.034037,0.000000,0.000862,0.000000,0.000862,0.0,0.000862,0.0,...,0.0,0.0,0.001723,0.0,0.000000,0.003878,0.000000,0.000862,0.000862,0.002154
2,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.0,0.012821,0.012821,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.012821,0.000000,0.000000,0.000000
4,0.0,0.010187,0.010187,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1269,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.001689,0.0,0.000000,0.006757,0.000000,0.000000,0.000000,0.000000
1270,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1271,0.0,0.000000,0.000000,0.003367,0.000000,0.003367,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.006734,0.000000,0.000000,0.000000,0.000000
1272,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [205]:
#把0值替换为极小值
eps = np.finfo(float).eps
num_word_df.iloc[num_word_df.iloc == 0] = eps
num_word_df

,spoiler,post,game,book,nezha,movie,soul,tang,character,rule,...,castorice,ban,might,someone,translation,wait,removed,amazing,definitely,shit
0,0.0,0.002232,0.033482,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.001116,0.0,0.002232,0.005580,0.000000,0.000000,0.000000,0.000000
1,0.0,0.000862,0.034037,0.000000,0.000862,0.000000,0.000862,0.0,0.000862,0.0,...,0.0,0.0,0.001723,0.0,0.000000,0.003878,0.000000,0.000862,0.000862,0.002154
2,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.0,0.012821,0.012821,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.012821,0.000000,0.000000,0.000000
4,0.0,0.010187,0.010187,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1269,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.001689,0.0,0.000000,0.006757,0.000000,0.000000,0.000000,0.000000
1270,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1271,0.0,0.000000,0.000000,0.003367,0.000000,0.003367,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.006734,0.000000,0.000000,0.000000,0.000000
1272,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## 3.数据整合为同一张表

In [206]:
fir = pd.concat([origin_df_normalized,pb],axis=1)
fir


,fans,vote,comment,综合热度,核心文化,载体表现,支撑机制,泛化讨论
推文1,0.015121,0.001097,0.006266,129.0,0.000309,0.331796,0.000115,0.667780
推文2,0.015121,0.014174,0.009400,935.0,0.161802,0.293599,0.000043,0.544556
推文3,0.015121,0.000067,0.010714,5.0,0.342029,0.347559,0.127699,0.182713
推文4,0.015121,0.000641,0.001314,53.0,0.123518,0.001543,0.000567,0.874372
推文5,0.015121,0.001755,0.002729,133.0,0.000460,0.767545,0.036569,0.195426
...,...,...,...,...,...,...,...,...
推文1270,0.002508,0.000101,0.000910,17.0,0.000560,0.734229,0.022485,0.242726
推文1271,0.000019,0.000000,0.010714,1.0,0.342029,0.347559,0.127699,0.182713
推文1272,0.002508,0.000641,0.000910,49.0,0.001001,0.827033,0.039311,0.132655
推文1273,0.000019,0.000000,0.010714,1.0,0.342029,0.347559,0.127699,0.182713


## 4.回归

In [208]:
import statsmodels.api as sm

# 2. 定义变量
y = fir["综合热度"]  # 因变量
X = fir[[
    "核心文化", "载体表现", "支撑机制",  # 主题变量（删除泛化讨论）
]]
X = sm.add_constant(X)  # 添加截距项

# 3. 构建Gamma回归模型
gamma_model = sm.GLM(
    y,
    X,
    family=sm.families.Gamma(link=sm.families.links.Log())  # 使用对数链接函数（Log）
)
result = gamma_model.fit()

# 4. 输出结果
print(result.summary())

ValueError: The first guess on the deviance function returned a nan.  This could be a boundary  problem and should be reported.